# 05 `step % accumulation_steps`：什么时候真的更新？

## 知识点标注

- 行业状态：常见工程实践。
- 现实用途：确认梯度累积是在第几个小批次执行参数更新，并发现最后不足一组时的边界问题。
- 本课类型：本地概念演示，不使用 GPU，不训练 MokioMind。

本课只观察一个问题：`accumulation_steps=2` 时，`step=1、2、3、4、5` 各自做什么？

## 先看规则

项目使用：

```python
if step % args.accumulation_steps == 0:
    scaler.step(optimizer)
    optimizer.zero_grad(set_to_none=True)
```

`%` 是取余数。`step % 2 == 0` 只在偶数步成立，所以第 2、4、6... 个小批次才触发更新。

In [1]:
accumulation_steps = 2
for step in range(1, 6):
    remainder = step % accumulation_steps
    action = '更新 + 清空梯度' if remainder == 0 else '只 backward，继续累积'
    print(f'step={step}: 余数={remainder} -> {action}')

step=1: 余数=1 -> 只 backward，继续累积
step=2: 余数=0 -> 更新 + 清空梯度
step=3: 余数=1 -> 只 backward，继续累积
step=4: 余数=0 -> 更新 + 清空梯度
step=5: 余数=1 -> 只 backward，继续累积


### 先预测

运行上面的代码前，先写下你的预测：

- 第 1 个小批次会更新吗？
- 第 2 个小批次会更新吗？
- 第 5 个小批次会更新吗？

## 5 个小批次时的边界

5 ÷ 2 = 2 组完整累积，还剩 1 个小批次。按照项目这条简单条件：

```text
第 1、2 批 -> 第 2 批更新一次
第 3、4 批 -> 第 4 批更新一次
第 5 批   -> 只累积，没有等到第 6 批
```

这不是数学错误，而是一个训练循环边界选择：如果希望最后剩余梯度也被使用，就需要额外处理 epoch 结束条件。

In [2]:
def update_steps(num_batches, accumulation_steps):
    return [
        step for step in range(1, num_batches + 1)
        if step % accumulation_steps == 0
    ]

for num_batches in [4, 5, 6]:
    print(f'{num_batches} 个小批次 -> 更新发生在第 {update_steps(num_batches, 2)} 批')

4 个小批次 -> 更新发生在第 [2, 4] 批
5 个小批次 -> 更新发生在第 [2, 4] 批
6 个小批次 -> 更新发生在第 [2, 4, 6] 批


## 对应 MokioMind 真实代码

- [trainer/train_pretrain.py:39](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:39)：逐个取出小批次，并从 `step=1` 开始编号。
- [trainer/train_pretrain.py:63](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:63)：每个小批次先把 loss 除以累积步数。
- [trainer/train_pretrain.py:65](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:65)：每个小批次都做 `backward()`。
- [trainer/train_pretrain.py:67](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:67)：只有整组累积完成才进入更新分支。
- [trainer/train_pretrain.py:76](/Users/mrwong/Documents/Program/AI/Minimind/eg/MokioMind-master/trainer/train_pretrain.py:76)：执行参数更新。

注意：项目当前代码在一个 epoch 最后不足 `accumulation_steps` 时，不会由这条条件自动补做一次更新。先理解这个行为，暂时不改项目代码。

## 验收题

1. `step=3`、`accumulation_steps=2` 时，为什么只 `backward()` 而不 `step()`？
2. 5 个小批次时，按当前条件会执行几次参数更新？第 5 批的梯度怎么办？
3. 这段逻辑解决的是参数更新时机，还是改变了一个小批次的形状？